In [1]:
from pathlib import Path
import random
from collections import defaultdict
import os

# -----------------------------------------------------
# Configuration
# -----------------------------------------------------
OUTPUT_FILE = "sampled_instances.txt"

SAMPLE_RATE = 0.20
RANDOM_SEED = 42

# Valid options
SET1_BIN_OPTIONS = ["THREE_TYPES", "FIVE_TYPES"]
SET1_COST_OPTIONS = ["LINEAR"]

SET2_BIN_OPTIONS = ["SEVEN_TYPES"]
SET2_COST_OPTIONS = ["LINEAR", "CONVEX", "CONCAVE"]


# -----------------------------------------------------
# Locate instance folders automatically
# -----------------------------------------------------
def find_instance_dirs():
    current = Path.cwd().resolve()

    for parent in [current] + list(current.parents):
        set1_candidates = list(parent.rglob("instances/set_1/N_1000"))
        set2_candidates = list(parent.rglob("instances/set_2/N_1000"))

        set1_candidates = [p for p in set1_candidates if p.is_dir()]
        set2_candidates = [p for p in set2_candidates if p.is_dir()]

        if set1_candidates and set2_candidates:
            print("Found instance folders:")
            print("Set 1:", set1_candidates[0])
            print("Set 2:", set2_candidates[0])
            return set1_candidates[0], set2_candidates[0]

    raise RuntimeError("Could not find instance folders automatically.")


SET1_DIR, SET2_DIR = find_instance_dirs()


# -----------------------------------------------------
# Produce relative path
# -----------------------------------------------------
def relative_path(path: Path) -> str:
    """
    Return a relative path from the current working directory.
    """
    return os.path.relpath(path.resolve(), Path.cwd().resolve())


# -----------------------------------------------------
# Parse instance names
# -----------------------------------------------------
def parse_set1_instance(path: Path):
    """
    Expected format:
        Correia_Random_x_y_z_t.txt

    Required:
        x = 4
        y = 3
        z in {9, 10, 11}
    """
    stem = path.stem
    parts = stem.split("_")

    if len(parts) != 6:
        return None

    prefix1, prefix2, x, y, z, t = parts

    if prefix1 != "Correia" or prefix2 != "Random":
        return None

    try:
        x = int(x)
        y = int(y)
        z = int(z)
        t = int(t)
    except ValueError:
        return None

    if x != 4:
        return None

    if y != 3:
        return None

    if z not in {9, 10, 11}:
        return None

    return {
        "set": "SET1",
        "name": path.name,
        "path": relative_path(path),
        "x": x,
        "y": y,
        "z": z,
        "t": t,
    }


def parse_set2_instance(path: Path):
    """
    Expected format:
        HS_Random_x_y_z.txt

    Required:
        x = 4
        y in {9, 10, 11}
    """
    stem = path.stem
    parts = stem.split("_")

    if len(parts) != 5:
        return None

    prefix1, prefix2, x, y, z = parts

    if prefix1 != "HS" or prefix2 != "Random":
        return None

    try:
        x = int(x)
        y = int(y)
        z = int(z)
    except ValueError:
        return None

    if x != 4:
        return None

    if y not in {9, 10, 11}:
        return None

    return {
        "set": "SET2",
        "name": path.name,
        "path": relative_path(path),
        "x": x,
        "y": y,
        "z": z,
        "t": "",
    }


# -----------------------------------------------------
# Build valid triples
# -----------------------------------------------------
def build_instance_triples():
    triples = []

    # -------------------- Set 1 --------------------
    for path in sorted(SET1_DIR.glob("*.txt")):
        parsed = parse_set1_instance(path)

        if parsed is None:
            continue

        for bin_option in SET1_BIN_OPTIONS:
            for cost_option in SET1_COST_OPTIONS:
                triples.append({
                    **parsed,
                    "bin_option": bin_option,
                    "cost_option": cost_option,
                })

    # -------------------- Set 2 --------------------
    for path in sorted(SET2_DIR.glob("*.txt")):
        parsed = parse_set2_instance(path)

        if parsed is None:
            continue

        for bin_option in SET2_BIN_OPTIONS:
            for cost_option in SET2_COST_OPTIONS:
                triples.append({
                    **parsed,
                    "bin_option": bin_option,
                    "cost_option": cost_option,
                })

    return triples


# -----------------------------------------------------
# Balanced strata
# -----------------------------------------------------
def stratum_key(row):
    """
    Balanced strata.

    Set 1:
        same number of samples for each:
            set, bin option, cost option, z

    Set 2:
        same number of samples for each:
            set, bin option, cost option, y
    """
    if row["set"] == "SET1":
        return (
            row["set"],
            row["bin_option"],
            row["cost_option"],
            row["z"],
        )

    if row["set"] == "SET2":
        return (
            row["set"],
            row["bin_option"],
            row["cost_option"],
            row["y"],
        )

    raise ValueError(f"Unknown set: {row['set']}")


def balanced_stratified_sample(rows, sample_rate, seed):
    rng = random.Random(seed)

    strata = defaultdict(list)

    for row in rows:
        strata[stratum_key(row)].append(row)

    total_rows = len(rows)
    target_total = max(1, round(sample_rate * total_rows))

    n_strata = len(strata)

    if n_strata == 0:
        return [], strata

    # Base number sampled per stratum.
    base = target_total // n_strata
    remainder = target_total % n_strata

    sampled = []

    stratum_items = sorted(strata.items(), key=lambda kv: kv[0])

    # Prefer giving the extra samples to larger strata.
    stratum_items_for_extra = sorted(
        stratum_items,
        key=lambda kv: len(kv[1]),
        reverse=True,
    )

    extra_keys = {
        key for key, _ in stratum_items_for_extra[:remainder]
    }

    for key, group in stratum_items:
        n_to_sample = base

        if key in extra_keys:
            n_to_sample += 1

        n_to_sample = min(n_to_sample, len(group))

        if n_to_sample > 0:
            sampled.extend(rng.sample(group, n_to_sample))

    sampled.sort(
        key=lambda r: (
            r["set"],
            r["bin_option"],
            r["cost_option"],
            r["z"] if r["set"] == "SET1" else r["y"],
            r["name"],
        )
    )

    return sampled, strata


# -----------------------------------------------------
# Save output for irace
# -----------------------------------------------------
def save_irace_instances_txt(rows, output_file):
    """
    Save one irace instance per line.

    Format:
        instance_path,bin_option,cost_option

    Example:
        ../instances/set_1/N_1000/Correia_Random_4_3_9_1.txt,THREE_TYPES,LINEAR
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                f"{row['path']},"
                f"{row['bin_option']},"
                f"{row['cost_option']}\n"
            )


# -----------------------------------------------------
# Main
# -----------------------------------------------------
def main():
    triples = build_instance_triples()

    if not triples:
        raise RuntimeError("No valid instance triples found.")

    sampled, strata = balanced_stratified_sample(
        triples,
        SAMPLE_RATE,
        RANDOM_SEED,
    )

    save_irace_instances_txt(sampled, OUTPUT_FILE)

    print("===== Balanced Stratified Sampling Summary =====")
    print(f"Total valid triples: {len(triples)}")
    print(f"Number of strata: {len(strata)}")
    print(f"Target sample size: {round(SAMPLE_RATE * len(triples))}")
    print(f"Actual sampled triples: {len(sampled)}")
    print(f"Sample rate: {SAMPLE_RATE:.0%}")
    print(f"Output file: {OUTPUT_FILE}")

    print("\n===== Strata sizes and sampled counts =====")

    sampled_by_key = defaultdict(int)
    for row in sampled:
        sampled_by_key[stratum_key(row)] += 1

    for key, group in sorted(strata.items(), key=lambda kv: kv[0]):
        print(
            f"{key}: total={len(group)}, "
            f"sampled={sampled_by_key[key]}"
        )

main()

Found instance folders:
Set 1: /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/instances/set_1/N_1000
Set 2: /home/pc-515/Documents/Graduate/Mestrado/Disciplinas/Período 2026-1/Métodos Híbridos em Otimização Combinatória/Programs/cos841_VSBPPC/instances/set_2/N_1000
===== Balanced Stratified Sampling Summary =====
Total valid triples: 150
Number of strata: 15
Target sample size: 30
Actual sampled triples: 30
Sample rate: 20%
Output file: sampled_instances.txt

===== Strata sizes and sampled counts =====
('SET1', 'FIVE_TYPES', 'LINEAR', 9): total=10, sampled=2
('SET1', 'FIVE_TYPES', 'LINEAR', 10): total=10, sampled=2
('SET1', 'FIVE_TYPES', 'LINEAR', 11): total=10, sampled=2
('SET1', 'THREE_TYPES', 'LINEAR', 9): total=10, sampled=2
('SET1', 'THREE_TYPES', 'LINEAR', 10): total=10, sampled=2
('SET1', 'THREE_TYPES', 'LINEAR', 11): total=10, sampled=2
('SET2', 'SEVEN_TYPES', 'CONCAVE', 9): total=10, sample